# CUDA Execution Model

This notebook explores what happens when a CPU process submits work to a GPU.

The main goal is to distinguish:

- kernel launch from kernel execution,
- asynchronous execution from synchronization,
- CPU wall-clock timing from GPU execution timing.

## Questions

1. Does Python wait when it launches a CUDA operation?
2. Why can normal Python timing produce misleading GPU measurements?
3. What does `torch.cuda.synchronize()` do?
4. How do CUDA events measure GPU execution time?
5. Which operations implicitly force the CPU to wait for the GPU?

## Mental model

A CUDA operation usually follows this path:

Python
↓
PyTorch
↓
CUDA runtime
↓
CUDA command queue
↓
GPU kernel execution

The CPU normally submits work to a CUDA stream and continues running.

Submitting the operation and completing the operation are separate events.

In [1]:
%pip install -q "modal>=1.5,<2"


[notice] A new release of pip is available: 23.2.1 -> 26.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import modal

LOCAL_PYTHON = f"{sys.version_info.major}.{sys.version_info.minor}"

app = modal.App("gpu-inference-systems-cuda-execution")

image = (
    modal.Image.debian_slim(python_version=LOCAL_PYTHON)
    .pip_install("torch")
)

In [4]:
# Define the first timing experiment
import time
import torch

@app.function(
    image=image,
    gpu="L4",
    timeout=300,
)
def measure_without_synchronization() -> dict[str, float]:
    size = 4096

    x = torch.randn(size, size, device="cuda")
    y = torch.randn(size, size, device="cuda")

    # Ensure tensor creation has completed before the experiment starts.
    torch.cuda.synchronize()

    start = time.perf_counter()

    z = x @ y

    launch_elapsed_ms = (time.perf_counter() - start) * 1000

    # We need the result to remain alive until after timing.
    del z

    torch.cuda.synchronize()

    return {
        "launch_elapsed_ms": launch_elapsed_ms,
    }

In [5]:
with modal.enable_output():
    with app.run():
        result = measure_without_synchronization.remote()

result

✓ Initialized. View run at 
https://modal.com/apps/vishalkt07/main/ap-zazX6W0RregONSyTuVaRRR
⠋ Initializing...
Building image im-0W8hc0ZYjgAY0cZHmm5gu0
m⠋ Creating objects...
=> Step 0: FROM base
m⠙ Creating objects...
=> Step 1: RUN python -m pip install torch
Looking in indexes: http://pypi-mirror.modal.local:5555/simple
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 508.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 531.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 519.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 525.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 538.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 505.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 540.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 523.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

{'launch_elapsed_ms': 75.79779300000044}

## Prediction

Will `launch_elapsed_ms` measure the complete matrix multiplication?

**My prediction:** No.

**Why:** `x @ y` submits the matrix-multiplication operation to a CUDA
stream and normally returns before the GPU finishes executing it.
The Python timer therefore measures host-side dispatch overhead, not
necessarily the complete GPU execution time.

## Warm-up and synchronized timing

The first matrix multiplication may perform one-time initialization.

This experiment will:

1. Run several warm-up matrix multiplications.
2. Measure the host-side dispatch duration.
3. Measure wall-clock duration after forcing synchronization.
4. Measure GPU execution using CUDA events.

In [13]:
@app.function(
    image=image,
    gpu="L4",
    timeout=300,
)
def compare_timing_methods() -> dict[str, float]:
    import time
    import torch
    
    size = 4096
    x = torch.randn(size, size, device="cuda")
    y = torch.randn(size, size, device="cuda")
    z = torch.empty(size, size, device="cuda")
    
    # Warm up the exact operation we intend to measure.
    for _ in range(5):
        torch.mm(x, y, out=z)

    torch.cuda.synchronize()

    # Host side launch timing
    launch_start = time.perf_counter()
    torch.mm(x, y, out=z)
    launch_elapsed_ms = (
        time.perf_counter() - launch_start
    ) * 1000
    torch.cuda.synchronize()

    # synchronized CPU wall-clock timing
    wall_start = time.perf_counter()
    torch.mm(x, y, out=z)
    torch.cuda.synchronize()
    synchronized_wall_ms = (
        time.perf_counter() - wall_start
    ) * 1000
    
    # CUDA event timing
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    start_event.record()
    torch.mm(x, y, out=z)
    end_event.record()
    end_event.synchronize()
    cuda_event_ms = start_event.elapsed_time(end_event)

    return {
        "launch_elapsed_ms": launch_elapsed_ms,
        "synchronized_wall_ms": synchronized_wall_ms,
        "cuda_event_ms": cuda_event_ms,
    }


In [11]:
with modal.enable_output():
    with app.run():
        timing_result = compare_timing_methods.remote()

timing_result

✓ Initialized. View run at 
https://modal.com/apps/vishalkt07/main/ap-R8QtQlGvqCNxVtoI475Zm3
⠋ Initializing...
⠴ Creating objects.....
├── 🔨 Created function measure_without_synchronization.
└── 🔨 Created function compare_timing_methods.
✓ Created objects.
├── 🔨 Created function measure_without_synchronization.
└── 🔨 Created function compare_timing_methods.
⠼ Worker assigned... View app at 
⠇ Worker assigned... View app at 
⠙ Running... View app at I475Zm3
⠼ Running... View app at I475Zm3
⠧ Loading images (1 containers initializing)... View app at 
⠋ Loading images (1 containers initializing)... View app at 
⠼ Loading images (1 containers initializing)... View app at 
⠧ Loading images (1 containers initializing)... View app at 
⠋ Loading images (1 containers initializing)... View app at 
⠸ Loading images (1 containers initializing)... View app at 
⠦ Loading images (1 containers initializing)... View app at 
⠋ Loading images (1 containers initializing)... View app at 
⠸ Running (1/1 con

{'launch_elapsed_ms': 0.08579799999974824,
 'synchronized_wall_ms': 11.810683999999405,
 'cuda_event_ms': 12.076031684875488}

## Observation

The measured results were:

- Host-side launch time: approximately 0.086 ms
- Synchronized wall-clock time: approximately 11.81 ms
- CUDA event time: approximately 12.08 ms

The host-side measurement was much smaller because `torch.mm` returned
after submitting the operation to the CUDA stream. It did not wait for
the GPU to finish.

The synchronized wall-clock measurement included both submission and
GPU completion because `torch.cuda.synchronize()` blocked the CPU.

The CUDA event measurement recorded elapsed time within the CUDA stream
and therefore more directly represented GPU execution time.

The similar synchronized and CUDA-event measurements confirm that the
matrix multiplication required approximately 12 ms, while submitting it
from the CPU required less than 0.1 ms.

## Conclusion

CUDA execution is asynchronous, not lazy.

`torch.mm(...)` submits work immediately. The GPU begins executing it
when earlier operations in the stream are complete, while the CPU is
normally free to continue.

Synchronization does not cause the operation to start. It causes the
CPU to wait until already-submitted GPU work has finished.